# Where Must a Fairness Fix Act? Adversarial Training versus Fine-Tuning for an Equal Child/Adult Decision in CLIP

**Isaac Mutuma · 202555404 · Deep Learning Programming 2026-2 · Track C (free topic), building on Track A (ViT fine-tuning) and Track B (adversarial training)**

> SKELETON (2026-11-20). Sections follow the final-submission rules: Problem/Prediction, Data/Model, Experiments, Results, Follow-up questions, Conclusion, plus the experiment log. `TODO` marks parts still to come. Ethics: no images of children are generated; UTKFace faces are never shown; real ICCWD photos appear only in §8, with children's eyes blurred on the displayed copies.

**How to run.** `RUN_FULL = False` (default) loads the saved result files and rebuilds every table and figure in a few minutes on a CPU. `RUN_FULL = True` retrains every model from the saved CLIP embeddings (GPU, several hours). Each section marks which code is **my implementation** and which is library or reference code.

## 0. Setup and reproducibility

In [ ]:
RUN_FULL = False          # True: retrain everything (GPU, hours); False: load saved results
SEED = 0
import json, math, platform, subprocess, pathlib, os
import numpy as np, pandas as pd
REPO = pathlib.Path('robust_child_filtering_model')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/isaacmutuma/robust_child_filtering_model.git', str(REPO)], check=True)
RES = REPO / 'results'
def load(rel): return json.loads((RES / rel).read_text())
print('python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__)
# TODO: torch / open_clip versions; Drive mount + sha256 checks of embeddings and folds when RUN_FULL

## 1. Problem, question and prediction

**Motivation.** Text-to-image training data is filtered with automated child detectors to reduce the risk of models generating child sexual abuse material (Crețu et al., 2026). In my earlier preregistered audit, all five detectors tested on UTKFace flagged Black minors as children less often than White minors (4.0 to 10.0 points after age adjustment); the gap sits among 10 to 17-year-olds, whose ages the detectors over-estimate. CLIP ViT-L/14 has the largest gap, and it comes from its image embeddings.

**Core question.** To make a CLIP-based child/adult decision at age 18 as accurate for Black teenagers as for White teenagers, is training on top of frozen CLIP features enough, or does the fix have to act inside the vision transformer?

**Prediction (recorded in the proposal before the experiments; not edited).** Adversarial training on frozen features (E) will have a small effect: a probe will still recover race from its representation, because the bias is in CLIP's embeddings, which E cannot change. Fine-tuning CLIP's last blocks with LoRA (F) has the best chance of narrowing the gap, but may overfit, since there are only 126 Black teenagers.

## 2. Data, model and baseline

- **Data:** UTKFace, 23,705 aligned faces with exact age and perceived-race labels (White, Black, Asian, Indian, Others). Minors (0–17) give TPR; adults 20–59 give FPR. Five fixed folds (4,741 images each, balanced by group × age band, seed 0); every image is predicted by a model that never saw it; settings are tuned on an inner fold.
- **Input / output / loss:** frozen CLIP ViT-L/14 (OpenAI weights, QuickGELU) → 768-number embedding → MLP head → age in years; child = predicted age < 18; L1 loss.
- **Metrics:** age-adjusted White − Black TPR gap (main), teen TPR by group, adult FPR, MAE, Black − White age error; paired bootstrap vs A (2,000 resamples); Holm across E, F1, F2.
- TODO: one figure of the data (counts by group × age band; no faces).

## 3. My implementation

TODO: inline, commented code for each piece (copied from the repo and checked to reproduce the repo's numbers):
- 3.1 Age head and training loop (model A) — mine
- 3.2 E: alternating adversary / encoder update, two targets — mine
- 3.3 Leakage probe (balanced accuracy) — mine; logistic regression from scikit-learn
- 3.4 LEACE eraser (follow-up 2) — mine, from the paper's closed form
- 3.5 LoRA adapters in CLIP's last block(s) (F1, F2) — TODO
- 3.6 Metrics and paired bootstrap — mine (ported from my audit)

In [ ]:
# TODO 3.1-3.6: inline implementation cells

## 4. Experiments

### 4.1 Model A (baseline)
1. *What I wonder:* does a head trained on UTKFace's own ages, on frozen CLIP features, keep the zero-shot gap?
2. *Prediction:* a gap still appears, probably smaller than zero-shot CLIP's 10 points.
3. *Change / fixed:* the baseline; grid of 8 settings, chosen by inner-fold MAE.

### 4.2 Model E (adversarial, invariant representation)
1. *What I wonder:* can adversarial training on frozen CLIP features remove the race information that makes Black teenagers look older, and close the gap at 18?
2. *Prediction:* small effect; the leakage probe will still recover race (see §1).
3. *Change / fixed:* add the adversary and alternating updates (5 adversary steps, lr 1e-2; targets: maximise error or base rates; λ ∈ {0.1, 0.3, 1, 3}); folds, loss, budget, tuning rule and evaluation as for A. Control: λ = 0 reproduces A exactly.

### 4.3 F1 (LoRA in CLIP's last block(s)) — TODO
### 4.4 F2 (F1 + E's adversary; ablation F2 vs F1) — TODO

In [ ]:
if RUN_FULL:
    raise NotImplementedError('TODO: training calls')
A, E = load('models/A.json'), load('models/E_inv.json')

## 5. Results

In [ ]:
def row(name, rep, paired=None):
    g, bg = rep['tpr_gap_W_minus_B'], rep['by_group']
    out = {'model': name, 'gap': f"{100*g['est']:.1f} ({100*g['ci'][0]:.1f} to {100*g['ci'][1]:.1f})",
           'teen TPR B / W': f"{bg['Black']['tpr_teen']['est']:.3f} / {bg['White']['tpr_teen']['est']:.3f}",
           'max adult FPR': f"{max(x['fpr_adult']['est'] for x in bg.values()):.3f}",
           'MAE': f"{rep['mae_overall']['est']:.2f}",
           'B-W age error': f"{rep['age_err_gap_B_minus_W']['est']:+.2f}"}
    if paired:
        r = paired['gap_reduction_A_minus_M']
        out['gap reduction vs A'] = f"{100*r['est']:+.1f} ({100*r['ci'][0]:+.1f} to {100*r['ci'][1]:+.1f})"
    return out
cmp_ = load('comparisons.json')
table = [row('A (baseline)', A['report']), row('E (adversarial)', E['report'], E['paired_vs_A'])]
# TODO: F1, F2 rows; Holm p from comparisons.json once complete
display(pd.DataFrame(table)); print('Holm family', cmp_['family'], '| interim' if not cmp_['complete'] else '')

TODO figures: TPR by age and group; predicted vs true age by group; E adversary accuracy per epoch; leakage probe bars (E vs A vs raw CLIP).

## 6. Re-check of the key comparison (3 seeds)

TODO: A vs the best model, seeds 0, 1, 2. ICCWD cannot re-check the gap (no age or race labels); it is used in §8 as a separate real-image test of overall child detection.

## 7. Follow-up questions

### 7.1 Follow-up 1: if the adversary is pushed harder, is race removed, and does the gap close?
*Observation it came from:* E drove its adversary to chance, yet a fresh probe read race from E's representation as well as from A's (0.816 vs 0.819).
*Prediction (recorded before the run):* the probe drops well below 0.82, age error passes +0.5 years, the gap shrinks far less than leakage, the Black − White over-estimation stays above 0.
*Result:* race was never removed (probe 0.81–0.82 at every λ up to 10); no significant gap change; age error +0.7 to +1.0 years at the strongest settings; over-estimation +1.7 to +2.2 years. The adversary is fooled rather than race removed, so the question stayed open → follow-up 2.

In [ ]:
fu1 = load('followups/E_force_race.json')
display(pd.DataFrame([{**row(n, r['report'], r['paired_vs_A']), 'probe': round(r['leakage_probe']['bal_acc']['model']['est'], 3)}
                      for n, r in fu1['settings'].items()]))

### 7.2 Follow-up 2: if race is removed from the embeddings by construction (LEACE), does the gap close?
*Observation it came from:* follow-up 1 could not remove race with an adversary.
*Prediction:* TODO (recorded before the run, see docs/FOLLOWUP_LEACE.md).
*Result:* TODO.

In [ ]:
# TODO: load('followups/LEACE.json') table + probe chart

## 8. Visual cases (real ICCWD photos)

TODO: a small hand-picked set of clearly benign ICCWD scenes run through A and the best model (prediction, decision, CLS attention map with layer, head, query token and aggregation stated). Children's eyes are blurred on the displayed copies only; the models run on the originals; no captions or URLs; images are loaded from Drive and never stored in git. Whether these images may appear in this notebook's saved outputs is awaiting Isaac's decision (ethics decision of 19 Nov covers the report and video). Overall ICCWD child-detection TPR/FPR for A vs the best model.

## 9. Failure cases, limits and conclusion

TODO. Context (not tested conditions): three earlier fixes on frozen features did not narrow the gap (below).

In [ ]:
ctx = {m: load(f'models/{m}.json') for m in ['B', 'C', 'C2']}
display(pd.DataFrame([row(m, d['report'], d['paired_vs_A']) for m, d in ctx.items()]))

## 10. Experiment log

Epochs, updates, batch size, seed, runtime and device for every run, generated from the result files.

In [ ]:
def log_rows(name, d, n_train=4 * 4741, batch=256):
    rows = []
    for f in d.get('folds', []):
        ep = f.get('epochs')
        rows.append({'run': name, 'test fold': f.get('test_fold'), 'chosen': f.get('chosen'),
                     'epochs': ep, 'updates': None if ep is None else ep * math.ceil(n_train / batch),
                     'batch': batch, 'seed': f"{SEED} + 1000*fold", 'seconds (tuning + refit)': f.get('seconds'),
                     'device': d.get('device'), 'commit': d['git']['commit'][:7]})
    return rows
log_df = pd.DataFrame(sum([log_rows('A', A), log_rows('E_inv', E)], []))
# TODO: follow-ups (fixed 60 epochs), F1, F2, seeds
display(log_df)

## 11. Sources and AI use

TODO: final list (proposal sources + Castleman & Korolova 2025; Wolfe et al. 2023; Belrose et al. 2023 LEACE; Elazar & Goldberg 2018). AI use: an AI coding assistant helped draft code and documents; I reviewed, ran and can explain all code; cells marked *my implementation* are the parts I changed and verified.